In [1]:
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as ctb

from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.feature_selection import SelectFromModel
from openfe import OpenFE, tree_to_formula, transform

import pandas as pd
from gensim.models import Word2Vec
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from pathlib import Path
from MetaFeature import getfeature
from sklearn.model_selection import StratifiedKFold

In [2]:
df = pd.read_csv('./dataset/data.csv')
target_cols = [f'target{i}' for i in range(7)]
# 标签编码成数字并记录映射关系
all_label_mapping = []
for col in target_cols:
    df[col], unique = pd.factorize(df[col])
    all_label_mapping.append(unique)

In [3]:
# 预处理数据
def preprocess(text):
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word.lower() not in stop_words]
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in filtered_tokens]
    return lemmatized_tokens
 
# 获取模型输入， 使用Word2Vec 将Target和DatasetName转换为编码
def get_input(df):
    return df.drop(['DatasetName', 'Target'], axis=1)

def get_word2vec(df, use_word2vec=True):
    for col_name in ['DatasetName', 'Target']:
        # 将所有数据集中的文本预处理并转换为列表的列表
        all_tokens = [preprocess(text) for text in df[col_name]]
        all_tokens = [[text] for text in df[col_name]]

        # 训练Word2Vec模型
        if use_word2vec:
            model = Word2Vec(all_tokens, vector_size=100, window=5, min_count=1, workers=4)
            model.train(all_tokens, total_examples=len(all_tokens), epochs=10)

            all_vectors = []
            for row in df[col_name].values:
                # 获取单词的向量表示
                word_vector = model.wv[row]
                all_vectors.append(word_vector)
            all_vectors = pd.DataFrame(all_vectors)
            all_vectors.columns = [f'{col_name}{i}' for i in all_vectors.columns]

        
    return all_vectors

fea_df = get_input(df)
w2v_df = get_word2vec(df)

# openfe

In [6]:
params = {"num_iterations": 1000, "seed": 42}
params.update(
    {
        'objective':'multiclass',
        "colsample_bytree": 0.8,
        "colsample_bynode": 0.8,
        "learning_rate": 0.01,
        "num_leaves":31,
        "verbose":-1
    }
)

def run_openfe(fea_df, col, params):

    task = 'classification'
    metric = 'multi_logloss'
    fea_df = fea_df.copy()
    fea_df = pd.concat([fea_df, fea_df]).reset_index(drop=True)
    x_df = fea_df.drop(target_cols, axis=1)
    y_df = fea_df[col]
    ntrain = int(x_df.shape[0]*0.5)
    train_x, val_x = x_df.iloc[:ntrain], x_df.iloc[ntrain:]

    train_idx, val_idx = pd.Series(train_x.index), pd.Series(val_x.index)
    # 训练openfe模型
    ofe = OpenFE()


    new_feature_list = ofe.fit(data=x_df, task=task, train_index=train_idx, val_index=val_idx, 
            metric=metric, label=y_df, seed=42,
            n_data_blocks=1,
            n_repeats=100,
            stage2_metric='random_importance',
            stage2_params=params)
    openfe_df, _ = transform(train_x, val_x, new_feature_list, n_jobs=8)
    # category列编码
    for col in openfe_df.select_dtypes(include=['category']).columns:
        openfe_df[col] = openfe_df[col].cat.codes
    return openfe_df, ofe, new_feature_list

In [7]:
# 设置task和metric
new_feature_dict = {}
all_pred_df = pd.DataFrame()
all_pred_df1, all_pred_df2, all_pred_df3, all_pred_df4 = pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
all_ofes = []
all_new_features = []
for col in target_cols:
    print(f'Processing {col}')

    openfe_df, ofe, new_feature_list = run_openfe(fea_df, col, params)
    all_ofes.append(ofe)
    all_new_features.append(new_feature_list)
    print(openfe_df.shape)
    
    n_fold = 5
    spliter = StratifiedKFold(n_splits=n_fold, shuffle=True, random_state=0)
    models = []
    y_pred_df1, y_pred_df2, y_pred_df3, y_pred_df4 = pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    for i, (train_index, test_index) in enumerate(spliter.split(openfe_df, openfe_df['type'])):
        def select_feature_train_predict(model, X, y, train_index, test_index):
            # selector = SelectFromModel(model, threshold='0.1*mean')
            # X = selector.fit_transform(X, y)
            X_train, y_train = X.loc[train_index], y.loc[train_index]
            X_test, y_test = X.loc[test_index], y.loc[test_index]
            model.fit(X_train, y_train)
            pred = model.predict(X_test)
            return pred
        
        y_df = fea_df[col]
        model1 = LogisticRegression()
        pred1 = select_feature_train_predict(model1, openfe_df.dropna(axis=1), y_df, train_index, test_index)

        model2 = RandomForestClassifier()
        pred2 = select_feature_train_predict(model2, openfe_df.dropna(axis=1), y_df, train_index, test_index)

        model3 = lgb.LGBMClassifier(**params)
        pred3 = select_feature_train_predict(model3, openfe_df, y_df, train_index, test_index)

        model4 = ctb.CatBoostClassifier()
        pred4 = select_feature_train_predict(model4, openfe_df, y_df, train_index, test_index)

        y_pred = (pred1 + pred2 + pred3 + pred4) / 4
        y_pred_df1 = pd.concat([y_pred_df1, pd.DataFrame(pred1, columns=[col])], axis=0)
        y_pred_df2 = pd.concat([y_pred_df2, pd.DataFrame(pred2, columns=[col])], axis=0)
        y_pred_df3 = pd.concat([y_pred_df3, pd.DataFrame(pred3, columns=[col])], axis=0)
        y_pred_df4 = pd.concat([y_pred_df4, pd.DataFrame(pred4, columns=[col])], axis=0)

    all_pred_df1[col] = y_pred_df1[col]
    all_pred_df2[col] = y_pred_df2[col]
    all_pred_df3[col] = y_pred_df3[col]
    all_pred_df4[col] = y_pred_df4[col]

Processing target0
The number of candidate features is 665
Start stage I selection.


100%|██████████| 4/4 [00:26<00:00,  6.66s/it]


192 same features have been deleted.
The number of remaining candidate features is 472
Start stage II selection.


100%|██████████| 4/4 [00:03<00:00,  1.29it/s]


Finish data processing.
Done with  100 of  100 (Spent   0.4 min)(300, 482)
Learning rate set to 0.073572
0:	learn: 1.4609656	total: 211ms	remaining: 3m 31s
1:	learn: 1.3357817	total: 261ms	remaining: 2m 10s
2:	learn: 1.2551166	total: 299ms	remaining: 1m 39s
3:	learn: 1.1729568	total: 344ms	remaining: 1m 25s
4:	learn: 1.1027185	total: 392ms	remaining: 1m 17s
5:	learn: 1.0359183	total: 440ms	remaining: 1m 12s
6:	learn: 0.9822096	total: 486ms	remaining: 1m 8s
7:	learn: 0.9345098	total: 529ms	remaining: 1m 5s
8:	learn: 0.8930808	total: 569ms	remaining: 1m 2s
9:	learn: 0.8532331	total: 607ms	remaining: 1m
10:	learn: 0.8178935	total: 646ms	remaining: 58.1s
11:	learn: 0.7815119	total: 689ms	remaining: 56.7s
12:	learn: 0.7564899	total: 727ms	remaining: 55.2s
13:	learn: 0.7279557	total: 769ms	remaining: 54.2s
14:	learn: 0.7061968	total: 811ms	remaining: 53.2s
15:	learn: 0.6806476	total: 849ms	remaining: 52.2s
16:	learn: 0.6618377	total: 901ms	remaining: 52.1s
17:	learn: 0.6453069	total: 942ms	r

...

Finish data processing.
Done with  100 of  100 (Spent   0.5 min)(300, 481)
Learning rate set to 0.073572
0:	learn: 1.3480501	total: 34.5ms	remaining: 34.4s
1:	learn: 1.3147129	total: 67.4ms	remaining: 33.6s
2:	learn: 1.2802020	total: 108ms	remaining: 35.9s
3:	learn: 1.2518645	total: 154ms	remaining: 38.4s
4:	learn: 1.2208780	total: 193ms	remaining: 38.3s
5:	learn: 1.1925842	total: 229ms	remaining: 37.9s
6:	learn: 1.1708313	total: 267ms	remaining: 37.9s
7:	learn: 1.1437694	total: 301ms	remaining: 37.3s
8:	learn: 1.1246559	total: 337ms	remaining: 37.1s
9:	learn: 1.1030232	total: 377ms	remaining: 37.3s
10:	learn: 1.0831211	total: 412ms	remaining: 37.1s
11:	learn: 1.0662497	total: 443ms	remaining: 36.5s
12:	learn: 1.0483652	total: 477ms	remaining: 36.2s
13:	learn: 1.0319403	total: 511ms	remaining: 36s
14:	learn: 1.0161882	total: 550ms	remaining: 36.1s
15:	learn: 0.9990391	total: 585ms	remaining: 36s
16:	learn: 0.9872545	total: 623ms	remaining: 36s
17:	learn: 0.9712128	total: 655ms	remainin

100%|██████████| 4/4 [00:30<00:00,  7.66s/it]


193 same features have been deleted.
The number of remaining candidate features is 472
Start stage II selection.


100%|██████████| 4/4 [00:03<00:00,  1.24it/s]


Finish data processing.
Done with  100 of  100 (Spent   0.6 min)(300, 482)
Learning rate set to 0.073572
0:	learn: 1.6783443	total: 54.8ms	remaining: 54.8s
1:	learn: 1.5942570	total: 118ms	remaining: 58.7s
2:	learn: 1.5266973	total: 179ms	remaining: 59.6s
3:	learn: 1.4569438	total: 225ms	remaining: 55.9s
4:	learn: 1.4016257	total: 270ms	remaining: 53.7s
5:	learn: 1.3531588	total: 316ms	remaining: 52.4s
6:	learn: 1.3056033	total: 368ms	remaining: 52.2s
7:	learn: 1.2638847	total: 417ms	remaining: 51.7s
8:	learn: 1.2281835	total: 473ms	remaining: 52.1s
9:	learn: 1.1899746	total: 518ms	remaining: 51.2s
10:	learn: 1.1547140	total: 566ms	remaining: 50.9s
11:	learn: 1.1309160	total: 608ms	remaining: 50.1s
12:	learn: 1.0978131	total: 657ms	remaining: 49.9s
13:	learn: 1.0648287	total: 703ms	remaining: 49.5s
14:	learn: 1.0361944	total: 750ms	remaining: 49.3s
15:	learn: 1.0133794	total: 795ms	remaining: 48.9s
16:	learn: 0.9880979	total: 841ms	remaining: 48.7s
17:	learn: 0.9649752	total: 890ms	rem

In [10]:
# i, j, k = 33, 33, 34
# all_pred_df = ((i*all_pred_df1 + j*all_pred_df2 + k*all_pred_df3)/100).round(0).astype(int)
all_pred_df = ((all_pred_df1 + 33*all_pred_df2 + 33*all_pred_df3 + all_pred_df4)/100).round(0).astype(int)
all_true_df = fea_df[target_cols]
metric_df = pd.DataFrame(data=0, index=target_cols, columns=['accuracy', 'precision', 'recall', 'f1'])
for col in target_cols:
    accuracy_score = classification_report(all_true_df[col], all_pred_df[col], output_dict=True)['accuracy']
    precision, recall, f1, _ = classification_report(all_true_df[col], all_pred_df[col], output_dict=True)['weighted avg'].values()
    metric_df.loc[col] += [accuracy_score, precision, recall, f1]
print("最优指标：")
print(metric_df.max())

最优指标：
accuracy     0.923333
precision    0.882564
recall       0.923333
f1           0.889261
dtype: float64


In [11]:
print(metric_df.mean())

accuracy     0.579048
precision    0.534436
recall       0.579048
f1           0.549036
dtype: float64


In [12]:
# # 遍历三个模型的结果加权
# f1_df = pd.DataFrame(columns=['i', 'j', 'k', 'max_f1', 'mean_f1'])
# for i in range(1, 100):
#     for j in range(1, 100):
#         for k in range(1, 100):
#             if i + j + k == 100:
#                 all_pred_df = ((i*all_pred_df1 + j*all_pred_df2 + k*all_pred_df3)/100).round(0).astype(int)
#                 all_true_df = fea_df[target_cols]
#                 f1_list = []
#                 for col in target_cols:
#                     accuracy_score = classification_report(all_true_df[col], all_pred_df[col], output_dict=True)['accuracy']
#                     precision, recall, f1, _ = classification_report(all_true_df[col], all_pred_df[col], output_dict=True)['weighted avg'].values()
#                     f1_list.append(f1)
#                 max_f1 = max(f1_list)
#                 mean_f1 = sum(f1_list) / len(f1_list)
#                 f1_df.loc[len(f1_df)] = [i, j, k, max_f1, mean_f1]
# f1_df       

In [39]:
target_name = 'Delay'
dataset_type = 0
dataset_name = 'airlines'

new_orign_df = pd.read_csv(f'./dataset/new_data/{dataset_name}.csv')
new_tensor = getfeature(new_orign_df)
new_numpy = new_tensor.mean(0).numpy()

new_df = pd.DataFrame(columns=[str(i) for i in range(7)])
new_df.loc[0] = new_numpy
new_df['DatasetName'] = dataset_name
new_df['Target'] = target_name
new_df['type'] = dataset_type
new_df['size0'] = new_df.shape[0]
new_df['size1'] = new_df.shape[1]
new_df = get_input(new_df)
y_pred_df = pd.DataFrame(columns=target_cols)

for col in target_cols:
    print(f'Processing {col}')

    new_x = new_df
    train_x = fea_df.drop(columns=target_cols)
    train_y = fea_df[col]
    openfe_df, _ = transform(train_x, new_x, new_feature_list, n_jobs=8)
    
    def select_feature_train_predict(model, train_x, test_x, train_y):
        selector = SelectFromModel(model, threshold='mean')
        train_x = selector.fit_transform(train_x, train_y)
        test_x = selector.transform(test_x)
        model.fit(train_x, train_y)
        pred = model.predict(test_x)
        return pred
        
    y_df = fea_df[col]
    model1 = LogisticRegression()
    pred1 = select_feature_train_predict(model1, train_x, new_x, train_y)

    model2 = RandomForestClassifier()
    pred2 = select_feature_train_predict(model2, train_x, new_x, train_y)

    model3 = lgb.LGBMClassifier(**params)
    pred3 = select_feature_train_predict(model3, train_x, new_x, train_y)

    model4 = ctb.CatBoostClassifier()
    pred4 = select_feature_train_predict(model4, train_x, new_x, train_y)

    y_pred = (pred1 + pred2 + pred3 + pred4) / 4
    y_pred_df.loc[dataset_name, col] = y_pred[0,0]
y_pred_df = y_pred_df.round(0).astype(int)
str_preds = []
for i, pred in enumerate(y_pred_df.loc[dataset_name]):
    label_mapping = all_label_mapping[i]
    pred = label_mapping[pred]
    str_preds.append(pred)
print(f'{dataset_name}的预测结果为{str_preds}')

Processing target0
Learning rate set to 0.074414
0:	learn: 1.4840349	total: 3.25ms	remaining: 3.25s
1:	learn: 1.3720837	total: 5.96ms	remaining: 2.97s
2:	learn: 1.2867460	total: 9.04ms	remaining: 3s
3:	learn: 1.2017403	total: 12ms	remaining: 2.99s
4:	learn: 1.1289654	total: 15.5ms	remaining: 3.09s
5:	learn: 1.0616670	total: 18.6ms	remaining: 3.08s
6:	learn: 0.9952840	total: 21.4ms	remaining: 3.04s
7:	learn: 0.9457904	total: 24.3ms	remaining: 3.02s
8:	learn: 0.8933397	total: 27.9ms	remaining: 3.08s
9:	learn: 0.8602010	total: 31.2ms	remaining: 3.09s
10:	learn: 0.8213223	total: 34.1ms	remaining: 3.06s
11:	learn: 0.7878606	total: 36.9ms	remaining: 3.04s
12:	learn: 0.7556400	total: 40.8ms	remaining: 3.1s
13:	learn: 0.7271692	total: 45ms	remaining: 3.17s
14:	learn: 0.6989088	total: 48.3ms	remaining: 3.17s
15:	learn: 0.6791966	total: 51.5ms	remaining: 3.17s
16:	learn: 0.6575414	total: 54.8ms	remaining: 3.17s
17:	learn: 0.6378920	total: 69.7ms	remaining: 3.8s
18:	learn: 0.6198726	total: 72.6ms

In [47]:
str_preds = []
for i, pred in enumerate(y_pred_df.loc[dataset_name]):
    label_mapping = all_label_mapping[i]
    pred = label_mapping[pred]
    str_preds.append(pred)
print(f'{dataset_name}的预测结果为{str_preds}')

airlines的预测结果为['imp_null', 'OE', 'ZS', 'fea_null', 'dup_null', 'out_null', 'RF']


In [48]:
target_name = 'usr'
dataset_type = 1
dataset_name = 'cpu_act'

new_orign_df = pd.read_csv(f'./dataset/new_data/{dataset_name}.csv')
new_tensor = getfeature(new_orign_df)
new_numpy = new_tensor.mean(0).numpy()

new_df = pd.DataFrame(columns=[str(i) for i in range(7)])
new_df.loc[0] = new_numpy
new_df['DatasetName'] = dataset_name
new_df['Target'] = target_name
new_df['type'] = dataset_type
new_df['size0'] = new_df.shape[0]
new_df['size1'] = new_df.shape[1]
new_df = get_input(new_df)
y_pred_df = pd.DataFrame(columns=target_cols)
for col in target_cols:
    print(f'Processing {col}')

    new_x = new_df
    train_x = fea_df.drop(columns=target_cols)
    train_y = fea_df[col]
    openfe_df, _ = transform(train_x, new_x, new_feature_list, n_jobs=8)
    
    def select_feature_train_predict(model, train_x, test_x, train_y):
        selector = SelectFromModel(model, threshold='mean')
        train_x = selector.fit_transform(train_x, train_y)
        test_x = selector.transform(test_x)
        model.fit(train_x, train_y)
        pred = model.predict(test_x)
        return pred
        
    y_df = fea_df[col]
    model1 = LogisticRegression()
    pred1 = select_feature_train_predict(model1, train_x, new_x, train_y)

    model2 = RandomForestClassifier()
    pred2 = select_feature_train_predict(model2, train_x, new_x, train_y)

    model3 = lgb.LGBMClassifier(**params)
    pred3 = select_feature_train_predict(model3, train_x, new_x, train_y)

    model4 = ctb.CatBoostClassifier()
    pred4 = select_feature_train_predict(model4, train_x, new_x, train_y)

    y_pred = (pred1 + pred2 + pred3 + pred4) / 4
    y_pred_df.loc[dataset_name, col] = y_pred[0,0]
y_pred_df = y_pred_df.round(0).astype(int)
str_preds = []
for i, pred in enumerate(y_pred_df.loc[dataset_name]):
    label_mapping = all_label_mapping[i]
    pred = label_mapping[pred]
    str_preds.append(pred)
print(f'{dataset_name}的预测结果为{str_preds}')

Processing target0
Learning rate set to 0.074414
0:	learn: 1.4840349	total: 3.23ms	remaining: 3.23s
1:	learn: 1.3720837	total: 5.92ms	remaining: 2.96s
2:	learn: 1.2867460	total: 8.48ms	remaining: 2.82s
3:	learn: 1.2017403	total: 11ms	remaining: 2.75s
4:	learn: 1.1289654	total: 13.4ms	remaining: 2.67s
5:	learn: 1.0616670	total: 16ms	remaining: 2.65s
6:	learn: 0.9952840	total: 18.4ms	remaining: 2.61s
7:	learn: 0.9457904	total: 21ms	remaining: 2.6s
8:	learn: 0.8933397	total: 23.4ms	remaining: 2.58s
9:	learn: 0.8602010	total: 25.9ms	remaining: 2.57s
10:	learn: 0.8213223	total: 28.4ms	remaining: 2.55s
11:	learn: 0.7878606	total: 30.9ms	remaining: 2.55s
12:	learn: 0.7556400	total: 33.2ms	remaining: 2.52s
13:	learn: 0.7271692	total: 35.7ms	remaining: 2.52s
14:	learn: 0.6989088	total: 38.1ms	remaining: 2.5s
15:	learn: 0.6791966	total: 40.9ms	remaining: 2.51s
16:	learn: 0.6575414	total: 43.6ms	remaining: 2.52s
17:	learn: 0.6378920	total: 46.2ms	remaining: 2.52s
18:	learn: 0.6198726	total: 48.7m